In [ ]:
!apt-get update
!apt-get install -y openjdk-11-jdk
!wget -qO - https://neo4j.com/artifact.php?name=neo4j-community-4.4.12-unix.tar.gz | tar xz
!mv neo4j-community-* neo4j
!sed -i 's/#dbms.default_listen_address/dbms.default_listen_address/' neo4j/conf/neo4j.conf
!sed -i 's/#dbms.connector.bolt.listen_address/dbms.connector.bolt.listen_address/' neo4j/conf/neo4j.conf

!echo "dbms.security.auth_enabled=false" >> neo4j/conf/neo4j.conf
!wget -P neo4j/plugins https://github.com/neo4j-contrib/neo4j-apoc-procedures/releases/download/4.4.0.7/apoc-4.4.0.7-all.jar
!echo "dbms.security.procedures.unrestricted=apoc.*" >> neo4j/conf/neo4j.conf
!echo "dbms.security.procedures.allowlist=apoc.*" >> neo4j/conf/neo4j.conf


In [ ]:
!neo4j/bin/neo4j start

In [ ]:
!pip install -q datasets pandas pymongo sentence_transformers
# !pip install -U transformers
# Install below if using GPU
!pip install -q accelerate
!pip install -q pymongo[srv]

In [ ]:
!pip install pyvis
import pyvis
print(pyvis.__version__)


In [ ]:
!pip install langchain_community
!pip install langchain_openai

In [ ]:
!pip install torch_geometric

In [ ]:
!pip install neo4j

In [ ]:
from langchain_community.graphs import Neo4jGraph

NEO4J_URI="bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "neo4jdlai"
NEO4J_DATABASE = "neo4j"

kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

In [ ]:
import openai

In [ ]:
from google.colab import userdata

In [ ]:
import pymongo
import pandas as pd
from google.colab import userdata
client = pymongo.MongoClient(userdata.get("mongo_url"))  # Update with your MongoDB connection string
db = client["hoanghamobilenew"]
collection = db["article_details_clean"]

In [ ]:
!gdown 1LgSkDWQhKx9KYZD-kpmDjgxJRaksVeSs

In [ ]:
df = pd.read_csv("/content/hoanghamobile.csv")

In [ ]:
# # Step 2: Retrieve data from MongoDB
# data = list(collection.find())

# # Step 3: Convert data to Pandas DataFrame
# df = pd.DataFrame(data)

# # Optional: Remove the MongoDB ObjectId column if it's not needed
# df.drop('_id', axis=1, inplace=True)


In [ ]:
# Display the DataFrame
df

In [ ]:
df

In [ ]:
import ast
def join_string(item):
    for i in range(len(item)):
        title, product_promotion, product_specs, current_price, color_options = item

        final_string = ""
        if title:
            final_string += f"{title}"

        if product_promotion:
            product_promotion = product_promotion.replace("<br>", " ").replace("\n", " ")
            final_string += f" {product_promotion}"

        if product_specs:
            product_specs = product_specs.replace("<br>", " ").replace("\n", " ")
            final_string += f" {product_specs}"

        if current_price:
            final_string += f" có giá: {current_price}"

        if color_options:
            final_string += " có màu sắc: "
            colors = ast.literal_eval(color_options)

            final_string += ", ".join(colors)


    return final_string

df['information'] = df[
    [
     'title',
     'product_promotion',
     'product_specs',
     'current_price',
     'color_options']
    ].astype(str).apply(join_string, axis=1)

# Display the DataFrame to confirm
df.head()

In [ ]:
df.head().iloc[0]["information"]

nokia 3210 4g - chính hãng
- Có thông tin sản phẩm: Công nghệ màn hình: IPS  Kích thước màn hình: 2.4 inch  Độ phân giải: 2MP  Hệ điều hành: S30+  Bộ nhớ trong: 128MB3  RAM: 64MB  Mạng di động: 2G, 3G, 4G, Hỗ trợ VoLTE2  Số khe SIM: Hai SIM Nano SIM + Nano SIM  Dung lượng pin: 1450mAh
- Giá hiện tại: 1,590,000 ₫
- Có màu sắc: Màu Vàng, Xanh, Màu Đen

In [ ]:
sample = df.iloc[4]["information"]

In [ ]:
sample

vivo y03 4/64gb- chính hãng - Ưu đãi tặng Sim Mobifone Hera & Key bản quyền Office 365 A3 khi mua điện thoại tại Hoàng Hà Mobile (SL có hạn)  Công nghệ màn hình: IPS LCD  Độ phân giải: HD+ (720 x 1612 Pixels), Chính 13 MP & Phụ 0.08 MP, 5 MP  Kích thước màn hình: 6.56 inch  Hệ điều hành: Funtouch OS 14.0  Vi xử lý: Helio G85  RAM: 4 GB  Mạng di động: Hỗ trợ 4G  Số khe SIM: 2 Nano SIM  Dung lượng pin: 5000 mAh  có giá: 2,790,000 ₫có màu sắc: Màu Đen, Xanh

In [ ]:
def extract_entities_and_relationships(text):
    # Initialize the OpenAI client

    from openai import OpenAI
    client = OpenAI(
        api_key = userdata.get("open_ai_key")
    )

    # text = sample

    prompt = (
        f"Extract entities (nodes) and their relationships (edges) from the text below."
        f"Entities and relationships MUST be in Vietnamese\n"
        f"Follow this format:\n\n"
        f"Entities:\n"
        f"- {{Entity}}: {{Type}}\n\n"
        f"Relationships:\n"
        f"- ({{Entity1}}, {{RelationshipType}}, {{Entity2}})\n\n"
        f"Text:\n\"{text}\"\n\n"
        f"Output:\nEntities:\n- {{Entity}}: {{Type}}\n...\n\n"
        f"Relationships:\n- ({{Entity1}}, {{RelationshipType}}, {{Entity2}})\n"
    )


    response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
        "role": "user",
        "content": [
            {
            "type": "text",
            "text": prompt
            }
        ]
        },
    ],
    response_format={
        "type": "text"
    },
    temperature=1,
    max_tokens=2048,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    )
    return response.choices[0].message.content

In [ ]:
result = extract_entities_and_relationships(sample)

In [ ]:
result

In [ ]:
def process_llm_out(result):
    import re

    # OpenAI response as a string
    response = result

    # Extract entities
    entity_pattern = r"- (.+): (.+)"
    entities = re.findall(entity_pattern, response)
    entity_dict = {entity.strip(): entity_type.strip() for entity, entity_type in entities}
    # { "Samsung Galaxy A100": "Sản phẩm", "Pin": "Thành phần"}
    entity_list = list(entity_dict.keys())

    # Extract relationships
    relationship_pattern = r"- \(([^,]+), ([^,]+), ([^)]+)\)"
    relationships = re.findall(relationship_pattern, response)
    relationship_list = [(subject.strip(), relation.strip().replace(" ", "_").upper(), object_.strip()) for subject, relation, object_ in relationships]

    # Output entities and relationships
    print("Entities:")
    for entity, entity_type in entity_dict.items():
        print(f"{entity}: {entity_type}")

    print("\nRelationships:")
    for subject, relation, object_ in relationship_list:
        print(f"({subject}, {relation}, {object_})")
    return entity_list, relationship_list

In [ ]:
entities, relas = process_llm_out(result)

In [ ]:
entities

In [ ]:
relas

In [ ]:
def add_relationships_to_neo4j(kg, relationships):
    """
    Add relationships extracted from the knowledge graph to the Neo4j database.
    """
    with kg._driver.session() as session:
        for subject, relation, obj in relationships:
            # Create nodes and relationships in Neo4j

            cypher_query = f"""
            MERGE (a:Entity {{name: $subject}})
            MERGE (b:Entity {{name: $object}})
            MERGE (a)-[:`{relation}`]->(b)
            """
            session.run(cypher_query, subject=subject, object=obj)
    print("Relationships added to Neo4j.")


In [ ]:
for i in range(30):
    sample = df.iloc[i]["information"]
    result = extract_entities_and_relationships(sample)
    _, relationship_list = process_llm_out(result)
    add_relationships_to_neo4j(kg, relationship_list)


In [ ]:
# results = kg.query(
#     """
#     MATCH (n)
#     DETACH DELETE n
#     """
# )


In [ ]:
add_relationships_to_neo4j(kg, relationship_list)

In [ ]:
results = kg.query(
"""MATCH (a)-[r]->(b)
RETURN a.name AS node_a, r, b.name AS node_b
""")
results

In [ ]:
results = kg.query(
"""MATCH (phone)-[:CÓ_DUNG_LƯỢNG_PIN]->(bateryCapaciy)
    WHERE phone.name = 'điện thoại poco c65'
RETURN bateryCapaciy.name
""")
results

In [ ]:
results = kg.query(
"""MATCH (phone)-[:CÓ_CÔNG_NGHỆ_MÀN_HÌNH]->(screenTech)
    WHERE screenTech.name = 'Dynamic AMOLED 2X'
RETURN phone.name
""")
results

In [ ]:

results = kg.query(
"""MATCH (device:Entity)-[:CÓ]->(km:Entity)-[:BẢO_HÀNH]->(value:Entity)
    WHERE device.name = 'điện thoại Tecno Camon 30'
RETURN value.name

""")
results

In [ ]:



results = kg.query(
"""MATCH (a:Entity)-[r]-(b)
    WHERE a.name = 'vivo y03' AND type(r) IN ['CÓ_DUNG_LƯỢNG_PIN', 'CÓ_SỐ_KHE_SIM', 'CÓ_RAM', 'CÓ_VI_XỬ_LÝ', 'CÓ_CAMERA_TRƯỚC']
RETURN a AS node_a, r AS relationship, b AS node_b;

""")
results

In [ ]:
CYPHER_GENERATION_TEMPLATE = """Task: Generate a Cypher statement to query a graph database.
Instructions:
- Analyze the question and extract relevant graph components dynamically. Use this to construct the Cypher query.
- Use only the relationship types and properties from the provided schema. Do not include any other relationship types, properties, or assumptions not defined in the schema.
- The schema is based on a graph structure with nodes and relationships as follows:
{schema}
- Return only the generated Cypher query in your response. Do not include explanations, comments, or additional text.
- Ensure the Cypher query directly addresses the given question using the schema accurately.

Examples:
# Thiết bị nào sử dụng công nghệ IPS?
# Dien thoai nao nào sử dụng công nghệ IPS?
MATCH (device)-[:CÓ_CÔNG_NGHỆ]->(techNode)
    WHERE techNode.name = 'IPS'
RETURN device.name

# Dung lượng pin của Nokia 3210 4G là gì?
MATCH (device)-[:CÓ_DUNG_LƯỢNG_PIN]->(batteryCapacity)
    WHERE device.name = 'Nokia 3210 4G'
RETURN batteryCapacity.name

# Mạng di động nào được thiết bị hỗ trợ?
MATCH (device)-[:HỖ_TRỢ_MẠNG]->(networkType)
    WHERE device.name = 'Nokia 3210 4G'
RETURN networkType.name

# Bộ nhớ trong của Nokia 3210 4G là gì?
MATCH (device:Entity)-[:CÓ_BỘ_NHỚ_TRONG]->(memory:Entity)-[:LÀ]->(value:Entity)
    WHERE device.name = 'Nokia 3210 4G'
RETURN value.name

# vivo y03 có ram mấy GB?
MATCH (device)-[:CÓ_RAM]->(ram)
    WHERE device.name = 'vivo y03'
RETURN ram.name

# Cho tôi cấu hình của vivo y03?
MATCH (a:Entity)-[r]-(b)
    WHERE a.name = 'vivo y03' AND type(r) IN ['CÓ_DUNG_LƯỢNG_PIN', 'CÓ_SỐ_KHE_SIM', 'CÓ_RAM', 'CÓ_VI_XỬ_LÝ', 'CÓ_CAMERA_TRƯỚC']
RETURN a AS node_a, r AS relationship, b AS node_b;

# Nói cho tôi biết thông tin về RAM của Honor X7b?
MATCH (device)-[:CÓ_RAM]->(ram)
    WHERE device.name = 'Honor X7b'
RETURN ram.name

# điện thoại poco c65 pin là bao nhiêu?
MATCH (phone)-[:CÓ_DUNG_LƯỢNG_PIN]->(bateryCapaciy)
    WHERE phone.name = 'điện thoại poco c65'
RETURN bateryCapaciy.name

# Samsung Galaxy S23 FE 5G pin thế nào nhỉ?
MATCH (phone)-[:CÓ_DUNG_LƯỢNG_PIN]->(bateryCapaciy)
    WHERE phone.name = 'Samsung Galaxy S23 FE 5G'
RETURN bateryCapaciy.name


# Những điện thoại có cùng màu đen với điện thoại poco c65?
MATCH (start1)-[:CÓ_MÀU_SẮC]->(sharedNode)<-[:CÓ_MÀU_SẮC]-(targetDevice)
    WHERE phone.name = 'điện thoại poco c65' AND sharedNode.name = 'Màu Đen'
RETURN targetDevice.name

The question is:
{question}
"""


In [ ]:
from dotenv import load_dotenv
import os

import textwrap

# Langchain
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain.chains import RetrievalQAWithSourcesChain
from langchain.prompts.prompt import PromptTemplate
from langchain.chains import GraphCypherQAChain
from langchain_openai import ChatOpenAI
# Warning control
import warnings
warnings.filterwarnings("ignore")

In [ ]:
CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template=CYPHER_GENERATION_TEMPLATE
)

In [ ]:
cypherChain = GraphCypherQAChain.from_llm(
    ChatOpenAI(
        temperature=0,
        openai_api_key=userdata.get("open_ai_key")
        ),
    graph=kg,
    verbose=True,
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    allow_dangerous_requests=True
)

In [ ]:
def prettyCypherChain(question: str) -> str:
    response = cypherChain.run(question)
    print('---response', response)
    print('===', cypherChain.invoke(question))
    print(textwrap.fill(response, 60))

In [ ]:
prettyCypherChain("Cấu hình của điện thoại poco c65 như thế nào?")

In [ ]:
results = kg.query(
"""MATCH (phone)-[:CÓ_MÀU_SẮC]->(colorNode)
    WHERE phone.name = 'điện thoại poco c65' AND colorNode.name = 'Màu Đen'
MATCH (phone)-[:CÓ_MÀU_SẮC]->(colorNode)<-[:CÓ_MÀU_SẮC]-(otherPhone)
    WHERE otherPhone <> phone
RETURN otherPhone.name
""")
results

Truy vấn với kết nối sâu

In [ ]:
results = kg.query(
"""MATCH path = (start:Entity {name: 'điện thoại poco c65'})-[*]->(end:Entity {name: 'Nokia 220 4G'})
RETURN last(nodes(path)).name AS lastNodeName
""")
results

In [ ]:
results = kg.query(
"""MATCH (start1:Entity {name: 'điện thoại poco c65'})-[:CÓ_MÀU_SẮC]->(sharedNode: Entity {name: 'Màu Đen'})<-[:CÓ_MÀU_SẮC]-(targetDevice)
RETURN targetDevice.name
"""
)

In [ ]:
results

In [ ]:
results = kg.query(
"""MATCH (device:Entity)-[:CÓ_BỘ_NHỚ_TRONG]->(value:Entity)
WHERE device.name = 'infinix note 40 pro'
RETURN value.name
"""
)
results

In [ ]:
def visualize_graph(data):
    try:
        import networkx as nx
        from pyvis.network import Network
        from IPython.display import IFrame, display

        # Create a directed graph
        G = nx.DiGraph()

        # Process the data to populate the graph
        for record in data:
            # Extract node_a, node_b, and the relationship type
            node_a = record.get('node_a')
            node_b = record.get('node_b')
            relationship = record.get('r')[1] if 'r' in record and isinstance(record['r'], tuple) else None

            if node_a and node_b and relationship:
                # Add nodes and edge with the relationship type as a label
                G.add_node(node_a, label=node_a)
                G.add_node(node_b, label=node_b)
                G.add_edge(node_a, node_b, label=relationship)

        # Create a Pyvis Network object with CDN resources set to 'in_line'
        net = Network(notebook=True, directed=True)
        net.from_nx(G)

        # Add edge labels for better visualization
        for edge in G.edges(data=True):
            net.add_edge(edge[0], edge[1], title=edge[2].get('label', ''))

        # Save the graph to an HTML file
        net.show('graph.html')

        # Embed the graph in Google Colab using IFrame
        display(IFrame('graph.html', width='100%', height='100%'))

    except Exception as e:
        print(f"Error while visualizing the graph: {str(e)}")
        return None


In [ ]:
visualize_graph(kg.query(
"""MATCH (a)-[r]->(b)
RETURN a.name AS node_a, r, b.name AS node_b
"""))

GCN

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password"))

def get_graph_data():
    with driver.session() as session:
        # Get nodes
        nodes_query = "MATCH (n:Entity) RETURN id(n) AS node_id, n.name AS name"
        nodes = session.run(nodes_query)
        node_mapping = {record["node_id"]: record["name"] for record in nodes}

        # Get edges and edge types
        edges_query = "MATCH (n)-[r]->(m) RETURN id(n) AS source, id(m) AS target, type(r) AS relationship_type"
        edges = session.run(edges_query)
        edge_list = [(record["source"], record["target"], record["relationship_type"]) for record in edges]

        # Extract all unique relationship types
        unique_relationship_types = {e[2] for e in edge_list}

        return node_mapping, edge_list, unique_relationship_types

node_mapping, edge_list, unique_relationship_types = get_graph_data()

# Print the unique relationship types
print("Unique Relationship Types:")
for rel_type in unique_relationship_types:
    print(rel_type)


In [ ]:
import torch
# Create a mapping for edge names to indices
edge_name_to_index = {name: idx for idx, name in enumerate(set(edge[2] for edge in edge_list))}

# Convert edge list to indices
edge_index = [(src, tgt, edge_name_to_index[name]) for src, tgt, name in edge_list]
edge_index = torch.tensor(edge_index, dtype=torch.long)
edge_index

In [ ]:
len(edge_index)

In [ ]:
edge_list

In [ ]:
node_mapping

In [ ]:
reverse_node_mapping = {value: key for key, value in node_mapping.items()}

### Only Node Embedding

In [ ]:
edge_list[0]

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import random

# Step 1: Create edge_index
edge_index = torch.tensor([[e[0], e[1]] for e in edge_list], dtype=torch.long).t().contiguous()

# Features for nodes (one-hot encoding)
num_nodes = len(node_mapping)
features = torch.eye(num_nodes)  # One-hot encoding for each node

# Split edges into training and validation sets
num_edges = edge_index.size(1)
indices = list(range(num_edges))
random.shuffle(indices)
split_idx = int(0.8 * num_edges)  # 80% for training, 20% for validation

train_indices = indices[:split_idx]
val_indices = indices[split_idx:]

train_edge_index = edge_index[:, train_indices]
val_edge_index = edge_index[:, val_indices]

# Step 2: Create Data objects for training and validation
train_data = Data(x=features, edge_index=train_edge_index)
val_data = Data(x=features, edge_index=val_edge_index)


In [ ]:
train_data.x

In [ ]:
edge_index

In [ ]:
features.size(0), features.size(1)

In [ ]:
edge_index.size(0), edge_index.size(1)

In [ ]:
src, tgt = edge_index

In [ ]:
# embeddings[src].shape

- N: Số lượng Node
- H: Hidden size
- E: Embedding size
- M: Số lượng kết nối

In [ ]:
# Step 3: Define Graph Autoencoder for Nodes Only
class GAE(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, embedding_dim):
        super(GAE, self).__init__()
        self.encoder1 = GCNConv(input_dim, hidden_dim)  # Initializes a GCN layer: (N, N) -> (N, H)
        self.encoder2 = GCNConv(hidden_dim, embedding_dim)  # Initializes a GCN layer: (N, H) -> (N, E)

    def encode(self, x, edge_index):
        # x: (N, N), edge_index: (2, M)
        x = F.relu(self.encoder1(x, edge_index))  # (N, N) -> (N, H)
        x = self.encoder2(x, edge_index)  # (N, H) -> (N, E)
        return x  # Node embeddings: (N, E)

    def decode(self, z, edge_index):
        # z: (N, E), edge_index: (2, M)
        src, tgt = edge_index  # src: (M,), tgt: (M,)
        return (z[src] * z[tgt]).sum(dim=1)  # (M, E) -> (M,)

    def forward(self, x, edge_index):
        # x: (N, F_in), edge_index: (2, M)
        z = self.encode(x, edge_index)  # Node embeddings: (N, E)
        reconstructed = self.decode(z, edge_index)  # Reconstruct edges: (M,)
        return z, reconstructed


In [ ]:
target = torch.ones(edge_index.size(1))

In [ ]:
target

In [ ]:
# Step 4: Initialize the model
input_dim = features.size(1)
hidden_dim = 16
embedding_dim = 8
model = GAE(input_dim, hidden_dim, embedding_dim)

# Step 5: Define optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
def loss_function(reconstructed, edge_index):
    # Binary cross-entropy loss for adjacency reconstruction
    # Một tensor toàn giá trị 1 với chiều dài bằng số lượng edges (M) trong đồ thị.
    target = torch.ones(edge_index.size(1))  # All edges exist
    # Sự khác biệt giữa logits dự đoán (reconstructed) và các giá trị thực (target).
    return F.binary_cross_entropy_with_logits(reconstructed, target)

# Step 6: Train the model and validate after each epoch
model.train()
for epoch in range(200):
    # Training phase
    optimizer.zero_grad()
    embeddings, reconstructed = model(train_data.x, train_data.edge_index)
    train_loss = loss_function(reconstructed, train_data.edge_index)
    train_loss.backward()
    optimizer.step()

    # Validation phase
    model.eval()
    with torch.no_grad():
        val_embeddings, val_reconstructed = model(val_data.x, val_data.edge_index)
        val_loss = loss_function(val_reconstructed, val_data.edge_index)

    # Switch back to train mode for next epoch
    model.train()

    # Print losses
    print(f'Epoch {epoch + 1}, Train Loss: {train_loss.item()}, Validation Loss: {val_loss.item()}')


In [ ]:
# reverse_node_mapping["Samsung Galaxy A05s"]

In [ ]:
node_mapping.values()

In [ ]:
query = "Điện thoại Samsung Galaxy A05s có giá bao nhiêu?"

In [ ]:
llm_res = extract_entities_and_relationships(query)

In [ ]:
 llm_res

In [ ]:
!pip install rapidfuzz

In [ ]:
query = "Nokia 3210 bên bạn giá bao nhiêu?"
entities, _ = process_llm_out(extract_entities_and_relationships(query))


In [ ]:
entities

In [ ]:
from rapidfuzz import process  # For fuzzy matching
import torch.nn.functional as F

def find_closest_entities(entities, node_mapping):
    """
    Finds the closest matching entities in node_mapping for a list of query entities.

    Parameters:
        entities (list): List of entity names to match.
        node_mapping (dict): Mapping of node IDs to entity names.

    Returns:
        list: A list of tuples [(query_entity, closest_match_id, closest_match_name, score)].
    """
    results = []
    node_names = list(node_mapping.values())
    for entity in entities:
        closest_match, score, index = process.extractOne(entity, node_names)
        closest_match_id = list(node_mapping.keys())[index]
        results.append((entity, closest_match_id, closest_match, score))
    return results


# Input: List of entities extracted from the query
query_entities = entities  # Replace with your entities

# Step 1: Find the closest matching nodes for all query entities
matches = find_closest_entities(query_entities, node_mapping)

print("Closest matches for query entities:")
for query_entity, match_id, match_name, score in matches:
    print(f"Query: '{query_entity}' -> Match: '{match_name}' (Node ID: {match_id}) with score {score:.2f}")

# Step 2: Use embeddings to find similar nodes for each matched entity
model.eval()
with torch.no_grad():
    embeddings, _ = model(train_data.x, train_data.edge_index)

# Normalize node embeddings
node_embeddings_norm = F.normalize(embeddings, p=2, dim=1)

# Step 3: Aggregate and analyze results for each matched entity
K = 20  # Number of top similar nodes to retrieve

for query_entity, match_id, match_name, score in matches:
    print(f"\nTop-{K} similar nodes for '{query_entity}' (Matched Node: {match_name}):")
    query_embedding = embeddings[match_id]
    query_embedding = F.normalize(query_embedding, p=2, dim=0)

    # Compute similarity scores
    similarity_scores = torch.matmul(query_embedding.unsqueeze(0), node_embeddings_norm.T).squeeze()

    # Retrieve Top-K similar nodes
    top_k_indices = torch.topk(similarity_scores, K).indices

    for idx in top_k_indices:
        similar_node_id = idx.item()
        similarity_score = similarity_scores[idx].item()
        similar_node_name = node_mapping[similar_node_id]

        # Check for direct connection in the edge list
        direct_connections = [
            e for e in edge_list if (e[0] == match_id and e[1] == similar_node_id) or
                                    (e[1] == match_id and e[0] == similar_node_id)
        ]

        # Print details in the desired format
        if direct_connections:
            for connection in direct_connections:
                source = node_mapping[connection[0]]
                target = node_mapping[connection[1]]
                relationship = connection[2]
                print(f"{source} -> {relationship} -> {target} with score {similarity_score:.4f}")
        else:
            # Print indirect connection paths
            # paths = find_indirect_connection(edge_list, node_mapping, match_id, similar_node_id)
            # if paths:
            #     print(f"Indirect paths between '{match_name}' and '{similar_node_name}':")
            #     for path in paths:
            #         formatted_path = " -> ".join(
            #             f"{node_mapping[src]} -[{rel}]-> {node_mapping[tgt]}" for src, rel, tgt in path
            #         )
            #         print(formatted_path)
            # else:
            print(f"{match_name} -> NO_DIRECT_RELATION -> {similar_node_name} with score {similarity_score:.4f}")


Combine queries

In [ ]:
from collections import deque

def find_indirect_connection(edge_list, node_mapping, start, target, max_depth=10):
    """
    Find indirect connections between two nodes using BFS.

    Parameters:
        edge_list (list): List of edges as (source, target, relationship).
        node_mapping (dict): Mapping of node IDs to node names.
        start (int): The starting node ID.
        target (int): The target node ID.
        max_depth (int): Maximum depth to explore for indirect connections.

    Returns:
        list: A list of paths from start to target.
    """
    # Build graph as adjacency list: {node: [(neighbor, relationship)]}
    graph = {}
    for src, tgt, rel in edge_list:
        if src not in graph:
            graph[src] = []
        if tgt not in graph:
            graph[tgt] = []
        graph[src].append((tgt, rel))
        graph[tgt].append((src, rel))  # Assuming undirected graph for traversal

    # Initialize BFS
    queue = deque([(start, [], 0)])  # (current_node, path_so_far, current_depth)
    visited = set()

    paths = []

    while queue:
        current_node, path, depth = queue.popleft()

        if depth > max_depth:  # Stop exploring if max depth is exceeded
            continue

        if current_node == target:  # Target node found
            paths.append(path)
            continue

        # Mark node as visited
        visited.add(current_node)

        # Explore neighbors
        for neighbor, relationship in graph.get(current_node, []):
            if neighbor not in visited:
                queue.append((neighbor, path + [(current_node, relationship, neighbor)], depth + 1))

    return paths

### Graph Walk Embedding

In [ ]:
# import torch

# def graph_walk_embedding(start_node, edge_index, embeddings, walk_length=2):
#     visited_nodes = set([start_node])  # Track visited nodes to avoid duplicates
#     current_nodes = [start_node]  # Start with the initial node

#     for _ in range(walk_length):
#         neighbors = []
#         for node in current_nodes:
#             # Find neighbors of the current node from edge_index
#             neighbors.extend(edge_index[1][edge_index[0] == node].tolist())
#         neighbors = list(set(neighbors))  # Remove duplicates
#         visited_nodes.update(neighbors)
#         current_nodes = neighbors

#     # Aggregate embeddings of visited nodes
#     visited_nodes = list(visited_nodes)  # Convert to list for indexing
#     visited_embeddings = embeddings[visited_nodes]
#     return visited_embeddings.mean(dim=0)

# # Example usage:
# # Example edge list
# edge_list = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (0, 5)]
# edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

# # Example node embeddings
# embeddings = torch.rand(6, 8)  # Random embeddings for 6 nodes, each of size 8

# # Define the start node and walk length
# start_node = 0
# walk_length = 3

# # Compute the query embedding
# query_embedding = graph_walk_embedding(start_node, edge_index, embeddings, walk_length)
# print("Query Embedding:", query_embedding)


In [ ]:
# # Normalize embeddings for cosine similarity
# embeddings_norm = torch.nn.functional.normalize(embeddings, p=2, dim=1)
# query_embedding_norm = torch.nn.functional.normalize(query_embedding.unsqueeze(0), p=2, dim=1)

# # Compute cosine similarity
# similarity_scores = torch.mm(query_embedding_norm, embeddings_norm.T).squeeze()

# # Retrieve Top-K similar nodes
# K = 3
# top_k_indices = torch.topk(similarity_scores, K).indices

# print("Top-K most similar nodes to the query:")
# for idx in top_k_indices:
#     print(f"Node {idx.item()} with similarity {similarity_scores[idx].item():.4f}")


Export data

In [ ]:
import json
def export_graph_to_json(kg, output_file="graph_data.json"):
    # Query to extract nodes
    nodes_query = "MATCH (n) RETURN id(n) AS node_id, labels(n) AS labels, properties(n) AS properties"
    nodes = kg.query(nodes_query)

    # Query to extract edges
    edges_query = "MATCH (n)-[r]->(m) RETURN id(n) AS source, id(m) AS target, type(r) AS relationship, properties(r) AS properties"
    edges = kg.query(edges_query)

    # Prepare data for export
    graph_data = {
        "nodes": nodes,
        "edges": edges
    }

    # Save data to JSON
    with open(output_file, "w") as f:
        json.dump(graph_data, f, indent=4)
    print(f"Graph data exported to {output_file}")

# Export graph to JSON
export_graph_to_json(kg)

In [ ]:
import json

def load_graph_from_json(input_file="graph_data.json"):
    # Load graph data from JSON
    with open(input_file, "r") as f:
        graph_data = json.load(f)

    # Extract nodes and edges
    nodes = graph_data["nodes"]
    edges = graph_data["edges"]

    return nodes, edges

# Load graph data
nodes, edges = load_graph_from_json()

# Example: Print loaded data
print("Loaded Nodes:")
for node in nodes:
    print(node)

print("\nLoaded Edges:")
for edge in edges:
    print(edge)
